# Issue with CMIP6 cesm2 slp dataset

When: 2026-02-02
Versions: clisops = , rook = 

There has been an error reported with the CMIP6 entry, which the CDS team quickly tested, here is the summary:

All requests I tested fail for the variable Sea level pressure and model CESM2 (USA).  The error says

`Process error: Resulting object does not have monotonic global indexes along dimension time.`

Tests for other random variables and models were fine.

I guess this is a Roocs error triggered by time subsetting, due to an issue with the time dimension in the SLP file from this model?


## Search Catalog

In [1]:
import intake

In [2]:
cat_url = "https://raw.githubusercontent.com/cp4cds/c3s_34g_manifests/master/intake/catalogs/c3s.yaml"

cat = intake.open_catalog(cat_url)
list(cat)

['c3s-cmip5',
 'c3s-cmip5-daily-pressure-level',
 'c3s-cmip5-daily-single-level',
 'c3s-cmip5-monthly-pressure-level',
 'c3s-cmip5-monthly-single-level',
 'c3s-cmip6',
 'c3s-cmip6-decadal',
 'c3s-cordex',
 'c3s-ipcc-atlas',
 'c3s-cica-atlas-v01',
 'c3s-cica-atlas']

In [3]:
df_cmip6 = cat['c3s-cmip6'].read()
df_cmip6.head()

,ds_id,path,size,mip_era,activity_id,institution_id,source_id,experiment_id,member_id,table_id,variable_id,grid_label,version,start_time,end_time,bbox,level
0,c3s-cmip6.ScenarioMIP.MOHC.UKESM1-0-LL.ssp245....,ScenarioMIP/MOHC/UKESM1-0-LL/ssp245/r1i1p1f2/A...,28037112,c3s-cmip6,ScenarioMIP,MOHC,UKESM1-0-LL,ssp245,r1i1p1f2,Amon,ts,gn,v20190507,2015-01-16T00:00:00,2049-12-16T00:00:00,"0.94, -89.38, 359.06, 89.38",NaN
1,c3s-cmip6.ScenarioMIP.MOHC.UKESM1-0-LL.ssp245....,ScenarioMIP/MOHC/UKESM1-0-LL/ssp245/r1i1p1f2/A...,38838222,c3s-cmip6,ScenarioMIP,MOHC,UKESM1-0-LL,ssp245,r1i1p1f2,Amon,ts,gn,v20190507,2050-01-16T00:00:00,2100-12-16T00:00:00,"0.94, -89.38, 359.06, 89.38",NaN
2,c3s-cmip6.ScenarioMIP.NCAR.CESM2.ssp370.r4i1p1...,ScenarioMIP/NCAR/CESM2/ssp370/r4i1p1f1/Amon/pr...,104081588,c3s-cmip6,ScenarioMIP,NCAR,CESM2,ssp370,r4i1p1f1,Amon,pr,gn,v20200528,2015-01-15T12:00:00,2064-12-15T12:00:00,"0.00, -90.00, 358.75, 90.00",NaN
3,c3s-cmip6.ScenarioMIP.NCAR.CESM2.ssp370.r4i1p1...,ScenarioMIP/NCAR/CESM2/ssp370/r4i1p1f1/Amon/pr...,74977662,c3s-cmip6,ScenarioMIP,NCAR,CESM2,ssp370,r4i1p1f1,Amon,pr,gn,v20200528,2065-01-15T12:00:00,2100-12-15T12:00:00,"0.00, -90.00, 358.75, 90.00",NaN
4,c3s-cmip6.ScenarioMIP.AS-RCEC.TaiESM1.ssp370.r...,ScenarioMIP/AS-RCEC/TaiESM1/ssp370/r1i1p1f1/Am...,144277888,c3s-cmip6,ScenarioMIP,AS-RCEC,TaiESM1,ssp370,r1i1p1f1,Amon,rlut,gn,v20201014,2015-01-16T12:00:00,2100-12-16T12:00:00,"0.00, -90.00, 358.75, 90.00",NaN


In [4]:
df = df_cmip6.loc[
    (df_cmip6.source_id=="CESM2" )
    & (df_cmip6.table_id=="Amon")
    & (df_cmip6.variable_id=="psl")
    & (df_cmip6.experiment_id=="ssp245")
]
df

,ds_id,path,size,mip_era,activity_id,institution_id,source_id,experiment_id,member_id,table_id,variable_id,grid_label,version,start_time,end_time,bbox,level
63254,c3s-cmip6.ScenarioMIP.NCAR.CESM2.ssp245.r4i1p1...,ScenarioMIP/NCAR/CESM2/ssp245/r4i1p1f1/Amon/ps...,67228932,c3s-cmip6,ScenarioMIP,NCAR,CESM2,ssp245,r4i1p1f1,Amon,psl,gn,v20200528,2015-01-15T12:00:00,2064-12-15T12:00:00,"0.00, -90.00, 358.75, 90.00",NaN
63255,c3s-cmip6.ScenarioMIP.NCAR.CESM2.ssp245.r4i1p1...,ScenarioMIP/NCAR/CESM2/ssp245/r4i1p1f1/Amon/ps...,48417760,c3s-cmip6,ScenarioMIP,NCAR,CESM2,ssp245,r4i1p1f1,Amon,psl,gn,v20200528,2065-01-15T12:00:00,2100-12-15T12:00:00,"0.00, -90.00, 358.75, 90.00",NaN


In [5]:
ds_ids = list(df.ds_id.unique())
ds_ids

['c3s-cmip6.ScenarioMIP.NCAR.CESM2.ssp245.r4i1p1f1.Amon.psl.gn.v20200528']

## Run subset operator

In [6]:
import os
os.environ['ROOK_URL'] = 'http://rook.dkrz.de/wps'

from rooki import rooki

### Request

In [7]:
resp = rooki.subset(
    collection=ds_ids,
    time='2050/2051',
    time_components='year:2050,2051|month:april',
    area='90,-180,-90,180'
)
resp.ok

True

In [8]:
resp

Metalink URL: http://rook7.cloud.dkrz.de:80/outputs/rook/4ccd64fa-0066-11f1-8e47-fa163eb671ca/input.meta4, num files: 1

In [9]:
resp.download_urls()


['http://rook7.cloud.dkrz.de:80/outputs/rook/4eb1ff2e-0066-11f1-89fc-fa163eb671ca/psl_Amon_CESM2_ssp245_r4i1p1f1_gn_20500415-20510415.nc']